In [1]:
import numpy as np
import pandas as pd

%pip install -q kaggle

import os, warnings, random, math, glob, time
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error
from sklearn.linear_model import LinearRegression, HuberRegressor
warnings.filterwarnings("ignore")

Note: you may need to restart the kernel to use updated packages.


In [2]:
SEARCH_ROOTS = ["/kaggle/input", "/content", "/content/drive", "/mnt", "."]
_candidates = []
for root in SEARCH_ROOTS:
    if os.path.isdir(root):
        _candidates += glob.glob(f"{root}/**/diabetes_subset_sensor_data", recursive=True)
assert _candidates, "Dataset not found — check it's attached."
DATA_DIR = os.path.dirname(sorted(_candidates, key=len)[0])
OUTPUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("DATA_DIR:", DATA_DIR)

DATA_DIR: /kaggle/input/datasets/sarabhian/d1namo-ecg-glucose-data


In [3]:
WINDOW_SIZE   = 48 #past 4 hours
HORIZONS      = [3, 6, 12]   # t+15, t+30, t+60 (5-min sampling)
SENSOR_COLS   = ["HR", "Activity", "PeakAccel",
                 "recent_carbs", "recent_units", "cgm_roc", "cgm_accel"]  # SkinTemp excluded — dead sensor
SKIN_TEMP_MIN, SKIN_TEMP_MAX = -40.0, 60.0
TOLERANCE_SEC = 90

#train-70%
#validation-10%
#testing-remaining 20%
TRAIN_FRAC, VAL_FRAC = 0.70, 0.10

#file handling
def resolve(*parts):
    p = os.path.join(DATA_DIR, *parts)
    inner = os.path.join(p, parts[0])
    return inner if os.path.isdir(inner) else p


#access sensor and glucose sets
subsets = {"diabetes": {"sensor": resolve("diabetes_subset_sensor_data"),
                        "glucose": resolve("diabetes_subset_pictures-glucose-food-insulin")}}

#this list will hold all the subjects found in the dataset (glucose and sensor data)
all_subjects = []
for group, paths in subsets.items():
    for subj_id in sorted(os.listdir(paths["sensor"])):
        sensor_path  = os.path.join(paths["sensor"], subj_id)
        glucose_path = os.path.join(paths["glucose"], subj_id, "glucose.csv")
        if os.path.isdir(sensor_path) and os.path.exists(glucose_path):
            all_subjects.append({"id": f"{group}_{subj_id}", "group": group,
                                  "sensor_path": sensor_path, "glucose_csv": glucose_path})

            print(f"Subjects found: {len(all_subjects)}")

Subjects found: 1
Subjects found: 2
Subjects found: 3
Subjects found: 4
Subjects found: 5
Subjects found: 6
Subjects found: 7
Subjects found: 8
Subjects found: 9


In [4]:
#preprocessing glucose and sensor (summary csv files)
def load_glucose(glucose_csv):
    df = pd.read_csv(glucose_csv)
    df = df[df["type"] == "cgm"].copy()
    df["datetime"] = pd.to_datetime(df["date"] + " " + df["time"])
    return df.sort_values("datetime").reset_index(drop=True)[["datetime", "glucose"]]

def load_sensor(sensor_path):
    sensor_dir = os.path.join(sensor_path, "sensor_data")
    if not os.path.isdir(sensor_dir):
        return None
    all_dfs = []
    for session in sorted(os.listdir(sensor_dir)):
        fpath = os.path.join(sensor_dir, session, f"{session}_Summary.csv")
        if os.path.exists(fpath):
            all_dfs.append(pd.read_csv(fpath))
    if not all_dfs:
        return None
    combined = pd.concat(all_dfs, ignore_index=True)
    combined["datetime"] = pd.to_datetime(combined["Time"], format="%d/%m/%Y %H:%M:%S.%f", errors="coerce")
    combined = combined.dropna(subset=["datetime"]).sort_values("datetime")
    base_cols = ["HR", "Activity", "PeakAccel"]
    cols = ["datetime"] + [c for c in base_cols if c in combined.columns]
    return combined[cols].reset_index(drop=True)

In [5]:
#only glucose
subject_merged = {}
for s in all_subjects:
    gdf = load_glucose(s["glucose_csv"])
    if len(gdf) < WINDOW_SIZE + max(HORIZONS):
        continue
    subject_merged[s["id"]] = gdf
    gdf["patient"] = s["id"]  # Add patient identifier

glucose_df = pd.concat(subject_merged.values(), ignore_index=True)
glucose_df #all patients all time

,datetime,glucose,patient
0,2014-10-01 19:14:00,10.3,diabetes_001
1,2014-10-01 19:19:00,9.9,diabetes_001
2,2014-10-01 19:24:00,9.8,diabetes_001
3,2014-10-01 19:29:00,9.6,diabetes_001
4,2014-10-01 19:34:00,9.4,diabetes_001
...,...,...,...
8050,2014-10-03 12:20:19,4.9,diabetes_009
8051,2014-10-03 12:25:19,4.2,diabetes_009
8052,2014-10-03 12:30:19,3.3,diabetes_009
8053,2014-10-03 12:35:19,2.7,diabetes_009


In [6]:
#find time frequency of glucose readings
glucose_df = glucose_df.sort_values("datetime")

glucose_df['time_diff'] = glucose_df["datetime"].diff()
glucose_df['time_diff'] = glucose_df['time_diff'].dt.total_seconds() / 60 #difference in minutes
 

#glucose_df['time_diff'].value_counts()

In [50]:
glucose_df.groupby("patient")["time_diff"].describe()

,count,mean,std,min,25%,50%,75%,max
patient,,,,,,,,
diabetes_001,1413.0,2.489797,1.058187,0.733333,1.983333,1.983333,3.900000,5.000000
diabetes_002,1056.0,0.636884,0.819556,0.100000,0.100000,0.100000,1.816667,1.933333
diabetes_003,183.0,1.317031,0.049353,1.233333,1.250000,1.350000,1.350000,1.350000
diabetes_004,969.0,1.046268,0.294430,0.933333,0.933333,0.933333,0.950000,3.033333
diabetes_005,909.0,0.114100,0.042262,0.083333,0.100000,0.100000,0.100000,0.266667
diabetes_006,1280.0,0.385130,0.434597,0.100000,0.116667,0.116667,1.100000,1.100000
diabetes_007,988.0,0.169939,0.313679,0.033333,0.050000,0.050000,0.050000,1.000000
diabetes_008,1139.0,2.479631,1.396431,1.600000,1.700000,1.716667,1.966667,5.000000
diabetes_009,117.0,0.109117,0.008332,0.100000,0.100000,0.116667,0.116667,0.116667


In [51]:
#glucose_df.loc[glucose_df['glucose'] < 3.9]  #hypoglycemia threshold
#glucose_df.loc[glucose_df['glucose'] > 11.1]  #hyperglycemia threshold

In [52]:
#only sensor readings
subject_merged = {}
for s in all_subjects:
    sdf = load_sensor(s["sensor_path"])
    if len(sdf) < WINDOW_SIZE + max(HORIZONS):
        continue
    subject_merged[s["id"]] = sdf
    sdf["patient"] = s["id"]  # Add patient identifier

sensor_df = pd.concat(subject_merged.values(), ignore_index=True)
sensor_df #all patients all time

,datetime,HR,Activity,PeakAccel,patient
0,2014-10-01 10:09:39.417,65,0.41,0.84,diabetes_001
1,2014-10-01 10:09:40.417,65,0.51,1.03,diabetes_001
2,2014-10-01 10:09:41.417,65,0.30,0.74,diabetes_001
3,2014-10-01 10:09:42.417,65,0.43,0.98,diabetes_001
4,2014-10-01 10:09:43.417,65,0.49,1.10,diabetes_001
...,...,...,...,...,...
1622568,2014-10-04 22:45:39.428,0,0.37,0.63,diabetes_009
1622569,2014-10-04 22:45:40.428,0,0.42,0.72,diabetes_009
1622570,2014-10-04 22:45:41.428,0,0.30,0.34,diabetes_009
1622571,2014-10-04 22:45:42.428,0,0.18,0.26,diabetes_009


In [53]:
#find time frequency of glucose readings
sensor_df = sensor_df.sort_values("datetime")

sensor_df['time_diff'] = sensor_df["datetime"].diff()
#sensor_df['time_diff'] = sensor_df['time_diff'].dt.total_seconds() / 60 #difference in minutes

In [34]:
sensor_df.groupby("patient")["time_diff"].describe()

,count,mean,std,min,25%,50%,75%,max
patient,,,,,,,,
diabetes_001,154486.0,0.001954,0.005127,0.000083,0.000083,0.000117,0.000133,0.016500
diabetes_002,167825.0,0.000662,0.237186,0.000000,0.000033,0.000050,0.000067,97.166700
diabetes_003,175962.0,0.000541,0.002864,0.000000,0.000017,0.000017,0.000050,0.016667
diabetes_004,157200.0,0.004282,0.006975,0.000133,0.000167,0.000200,0.016133,0.016250
diabetes_005,168621.0,0.000026,0.000029,0.000000,0.000017,0.000017,0.000033,0.000300
diabetes_006,201255.0,0.014102,0.005392,0.000133,0.016033,0.016133,0.016267,0.016667
diabetes_007,214295.0,0.000111,0.000068,0.000000,0.000067,0.000083,0.000133,0.000417
diabetes_008,193637.0,0.004225,0.007130,0.000000,0.000083,0.000100,0.016467,0.016667
diabetes_009,189291.0,0.002433,0.559264,0.000000,0.000050,0.000067,0.000183,243.316967


In [60]:
sensor_df.loc[sensor_df['HR'] == 0]

,datetime,HR,Activity,PeakAccel,patient,time_diff
1433288,2014-10-01 05:59:37.427,0,0.20,0.31,diabetes_009,0 days 00:00:01
1433289,2014-10-01 05:59:38.427,0,0.53,0.75,diabetes_009,0 days 00:00:01
1433290,2014-10-01 05:59:39.427,0,0.58,0.96,diabetes_009,0 days 00:00:01
1433301,2014-10-01 05:59:50.427,0,0.11,0.51,diabetes_009,0 days 00:00:01
1433302,2014-10-01 05:59:51.427,0,0.34,0.38,diabetes_009,0 days 00:00:01
...,...,...,...,...,...,...
1622568,2014-10-04 22:45:39.428,0,0.37,0.63,diabetes_009,0 days 00:00:01
1622569,2014-10-04 22:45:40.428,0,0.42,0.72,diabetes_009,0 days 00:00:01
1622570,2014-10-04 22:45:41.428,0,0.30,0.34,diabetes_009,0 days 00:00:01
1622571,2014-10-04 22:45:42.428,0,0.18,0.26,diabetes_009,0 days 00:00:01


### Combined Glucose and Sensor data by time

In [13]:
#align sensor and glucose data based on timestamps
def align_glucose_sensor(glucose_df, sensor_df):
    merged = pd.merge_asof(
        glucose_df.sort_values("datetime"), sensor_df.sort_values("datetime"),
        on="datetime", tolerance=pd.Timedelta(seconds=TOLERANCE_SEC), direction="nearest")
    return merged.dropna(subset=["glucose"]).reset_index(drop=True)

In [14]:
subject_merged = {}
for s in all_subjects:
    gdf = load_glucose(s["glucose_csv"]); sdf = load_sensor(s["sensor_path"])
    sdf["patient"] = s["id"]  # Add patient identifier
    if sdf is None or len(gdf) < WINDOW_SIZE + max(HORIZONS):
        continue
    merged = align_glucose_sensor(gdf, sdf)   #for one subject at a time
    subject_merged[s["id"]] = merged

subj_ids = list(subject_merged.keys())
print("Subjects with usable data:", len(subj_ids))

Subjects with usable data: 9


In [15]:
merged_df = pd.concat(subject_merged.values(), ignore_index=True)
merged_df

,datetime,glucose,HR,Activity,PeakAccel,patient
0,2014-10-01 19:14:00,10.3,68.0,0.01,0.04,diabetes_001
1,2014-10-01 19:19:00,9.9,87.0,0.01,0.06,diabetes_001
2,2014-10-01 19:24:00,9.8,75.0,0.06,0.10,diabetes_001
3,2014-10-01 19:29:00,9.6,93.0,0.13,0.22,diabetes_001
4,2014-10-01 19:34:00,9.4,95.0,0.24,0.54,diabetes_001
...,...,...,...,...,...,...
8050,2014-10-03 12:20:19,4.9,0.0,0.06,0.08,diabetes_009
8051,2014-10-03 12:25:19,4.2,0.0,0.03,0.06,diabetes_009
8052,2014-10-03 12:30:19,3.3,36.0,0.24,0.26,diabetes_009
8053,2014-10-03 12:35:19,2.7,34.0,0.08,0.19,diabetes_009


In [16]:
merged_df = merged_df.sort_values("datetime")

merged_df['time_diff'] = merged_df["datetime"].diff()
merged_df['time_diff'] = merged_df['time_diff'].dt.total_seconds() / 60 #difference in minutes
 

In [17]:
merged_df.groupby("patient")["time_diff"].describe()

,count,mean,std,min,25%,50%,75%,max
patient,,,,,,,,
diabetes_001,409.0,1.864466,0.364651,0.733333,1.983333,1.983333,1.983333,1.983333
diabetes_002,483.0,0.548516,0.765109,0.100000,0.100000,0.100000,1.800000,1.933333
diabetes_003,161.0,1.326190,0.045480,1.233333,1.350000,1.350000,1.350000,1.350000
diabetes_004,502.0,1.136687,0.377888,0.933333,0.933333,0.933333,0.950000,3.033333
diabetes_005,551.0,0.123261,0.052299,0.083333,0.083333,0.100000,0.116667,0.266667
diabetes_006,493.0,0.124983,0.019263,0.100000,0.116667,0.116667,0.133333,0.166667
diabetes_007,612.0,0.143301,0.281753,0.033333,0.050000,0.050000,0.050000,1.000000
diabetes_008,647.0,1.826662,0.585366,1.600000,1.700000,1.716667,1.716667,5.000000
diabetes_009,115.0,0.109130,0.008331,0.100000,0.100000,0.116667,0.116667,0.116667


could time parse to every 5 min, 10 min, ...